## 0. Colab GPU and transfer settings

Choose a GPU runtime. Use the Files pane to upload the approved ZIP to /content or mount Drive and edit PACKAGE. Verify authorization before uploading. Record your downstream overlap acknowledgment before training.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from importlib import metadata
import bisect
import csv
import hashlib
import itertools
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import uuid

import tempfile
import zipfile

from google.colab import drive, files
MOUNT_DRIVE = False
PACKAGE = Path("/content/lead_pd_preprocessed_seed0.zip")
OUTPUT_ROOT = Path("/content/lead_pd_runs")
CLOUD_TRANSFER_AUTHORIZED = False
OVERLAP_ACKNOWLEDGMENT = ""
BATCH_SIZE = 32
ACCUMULATION = 1
RESUME = False
if not CLOUD_TRANSFER_AUTHORIZED:
    raise RuntimeError("Confirm applicable cloud transfer authorization first.")
if MOUNT_DRIVE:
    drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mne==1.13.2", "einops==0.8.2",
                "huggingface_hub==1.33.0", "scikit-learn==1.6.1", "PyYAML"], check=True)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn import metrics
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU in Runtime > Change runtime type.")
CUDA = True
DEVICE = torch.device("cuda")
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0),
      round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GiB VRAM")
PROJECT = OUTPUT_ROOT
for folder in ("logs", "checkpoints", "results"):
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
versions = {name: metadata.version(name) for name in
            ("torch", "mne", "einops", "numpy", "pandas", "scipy", "scikit-learn", "huggingface_hub")}
def atomic(path, writer):
    path = Path(path)
    if path.is_relative_to(Path("/content/drive")):
        # Stage on local disk to avoid mounted Drive temporary-file errors.
        with tempfile.TemporaryDirectory(dir="/content") as staging:
            temporary = Path(staging) / path.name
            writer(temporary)
            shutil.copyfile(temporary, path)
        return
    temporary = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        writer(temporary)
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

def write_json(path, value):
    atomic(path, lambda p: p.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8"))

def write_table(path, table, separator="\t"):
    atomic(path, lambda p: table.to_csv(p, index=False, sep=separator))

def sha_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

MEDLEY_COMMIT = "d5bf678556ab9aeb0acf44ba2e7f0c5ad1c3b6bb"
LEAD_COMMIT = "dddee9c94f13b83d76f41ac9d452a74f84a76c5b"
HF_FILE = ("checkpoints/LEAD/pretrain/LEAD/P-11-b1024-p50-g16-learnable-3D/"
           "nh8_el12_dm128_df256_seed41/checkpoint.pth")
WEIGHTS = OFFICIAL / HF_FILE

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True



## 1. Validate and extract the transfer package

Validate checksums, file names, array shapes and split integrity. Extract recording arrays to local runtime disk, including when the archive lives in Drive.

In [ ]:

def read_transfer(package, destination):
    if not package.is_file():
        raise FileNotFoundError(f"Upload the approved ZIP and set PACKAGE: {package}")
    with zipfile.ZipFile(package) as z:
        names = z.namelist()
        if len(names) != len(set(names)) or not {"manifest.json", "manifest.sha256"}.issubset(names):
            raise ValueError("Missing manifest or duplicate ZIP members.")
        payload = z.read("manifest.json")
        if hashlib.sha256(payload).hexdigest() != z.read("manifest.sha256").decode().strip():
            raise ValueError("Manifest checksum mismatch.")
        bundle = json.loads(payload)
        if (bundle["format_version"], bundle["layout"], bundle["dtype"], bundle["normalized"],
                bundle["medley"], bundle["lead"]) != (1, "N,T,C", "float32", False, MEDLEY_COMMIT, LEAD_COMMIT):
            raise ValueError("Incompatible transfer manifest.")
        arrays = bundle["sha256"]
        if set(names) != ({"manifest.json", "manifest.sha256"} | set(arrays)) or any(
                not re.fullmatch(r"windows/recording-[0-9]{4,}\.npy", n) for n in arrays):
            raise ValueError("Unexpected ZIP contents.")
        subjects = pd.DataFrame(bundle["subjects"])
        expected = {"subject_uid", "person_uid", "cohort", "label_pd", "split"}
        if set(subjects) != expected or subjects.subject_uid.duplicated().any() or not (
                subjects.subject_uid.str.fullmatch(r"s-[a-f0-9]{32}").all()
                and subjects.person_uid.str.fullmatch(r"p-[a-f0-9]{32}").all()
                and set(subjects.cohort) == {"turku", "unm", "ucsd"}
                and set(subjects.split) == {"train", "val", "test"}
                and subjects.label_pd.isin([0, 1]).all()
                and subjects.groupby("split").label_pd.nunique().eq(2).all()
                and subjects.groupby("person_uid").split.nunique().eq(1).all()
                and subjects.groupby("person_uid").label_pd.nunique().eq(1).all()):
            raise ValueError("Invalid participant table or split.")
        records = pd.DataFrame(bundle["recordings"])
        if (set(records) != {"subject_uid", "array", "n_windows", "flagged_offsets"}
                or records.array.duplicated().any() or set(records.array) != set(arrays)
                or set(records.subject_uid) != set(subjects.subject_uid)):
            raise ValueError("Recording index does not match participants or archive files.")
        plan = []
        for r in records.itertuples(index=False):
            if (type(r.n_windows) is not int or r.n_windows < 1
                    or r.flagged_offsets != sorted(set(r.flagged_offsets))
                    or any(type(f) is not int or not 0 <= f < r.n_windows for f in r.flagged_offsets)):
                raise ValueError(f"Invalid window or flag indices for {r.array}")
            target = destination / r.array
            info = z.getinfo(r.array)
            expected_size = r.n_windows * 400 * 19 * 4
            if not expected_size < info.file_size <= expected_size + 1024:
                raise ValueError(f"Unexpected file size for {r.array}")
            cached = target.is_file() and sha_file(target) == arrays[r.array]
            plan.append((r, target, info, cached))
        needed = sum(info.file_size for _, _, info, cached in plan if not cached)
        if shutil.disk_usage(destination).free < needed + 128 * 2**20:
            raise OSError("Insufficient runtime scratch space.")
        for r, target, info, cached in plan:
            if not cached:
                def extract_one(temporary):
                    with z.open(info) as source, temporary.open("wb") as output:
                        shutil.copyfileobj(source, output, length=1024*1024)
                    if sha_file(temporary) != arrays[r.array]:
                        raise ValueError(f"Checksum mismatch: {r.array}")
                atomic(target, extract_one)
            x = np.load(target, mmap_mode="r", allow_pickle=False)
            if x.shape != (r.n_windows, 400, 19) or x.dtype != np.float32:
                raise ValueError(f"Invalid EEG array: {r.array}")
            for start in range(0, len(x), 128):
                if not np.isfinite(x[start:start+128]).all():
                    raise ValueError(f"Non-finite EEG: {r.array}")
            del x
        records["cache_path"] = records.array.map(lambda n: str(destination / n))
        records["cache_key"] = records.array.map(arrays)
        records = records.merge(subjects, on="subject_uid", validate="many_to_one")
    return bundle, subjects, records, hashlib.sha256(payload).hexdigest()

EXTRACTED = Path("/content/lead_pd_transfer")
(EXTRACTED / "windows").mkdir(parents=True, exist_ok=True)
bundle, finalized, cache_rows, TRANSFER_SHA = read_transfer(PACKAGE, EXTRACTED)
CFG = dict(bundle["training"], batch_size=BATCH_SIZE, accumulation=ACCUMULATION,
           resume=RESUME, overlap_acknowledgment=OVERLAP_ACKNOWLEDGMENT, amp=False)
seed_all(CFG["seed"])
print(bundle["audit"])


## 2. Fetch pinned source and pretrained weights

Fetch the audited public code and P-Base. Verify commits, record the weight digest and compare the preprocessing contract. Network or compatibility errors stop the run.

In [ ]:

MEDLEY = Path("/content/lead_pd_repos/Medley_skkai")
OFFICIAL = MEDLEY.parent / "LEAD"
MEDLEY.parent.mkdir(exist_ok=True)
def check_checkout(repo, commit):
    if not (repo / ".git").exists():
        raise FileNotFoundError(f"Clone the reference repository to {repo} and check out {commit}")
    def git(*args):
        return subprocess.check_output(["git", "-C", str(repo), *args], text=True).strip()
    if git("rev-parse", "HEAD") != commit or git("status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError(f"Expected clean tracked source at {commit} in {repo}")


for path, url, commit in (
    (MEDLEY, "https://github.com/jeongwon8u8/Medley_skkai", MEDLEY_COMMIT),
    (OFFICIAL, "https://github.com/DL4mHealth/LEAD", LEAD_COMMIT)):
    if not path.exists():
        subprocess.run(["git", "clone", url, str(path)], check=True)
    head = subprocess.check_output(["git", "-C", str(path), "rev-parse", "HEAD"], text=True).strip()
    if head != commit:
        if path == MEDLEY:
            subprocess.run(["git", "-C", str(path), "fetch", "origin", "refs/pull/3/head"], check=True)
        subprocess.run(["git", "-C", str(path), "checkout", "--detach", commit], check=True)
    check_checkout(path, commit)
from huggingface_hub import hf_hub_download
WEIGHTS = Path(hf_hub_download(repo_id="YiheWang/LEAD", filename=HF_FILE, local_dir=OFFICIAL))
if not WEIGHTS.is_file():
    raise FileNotFoundError(f"Place the official P-Base weights at {WEIGHTS}")
WEIGHT_SHA = sha_file(WEIGHTS)
MEDLEY_PY = MEDLEY / "medley_pd"
for repo in (MEDLEY_PY, OFFICIAL):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
from medley import eeg as medley_eeg
from medley.encoders.lead import LEAD as FrozenLEAD, _zscore
from models import LEAD as official_lead

if not Path(medley_eeg.__file__).is_relative_to(MEDLEY_PY) or not Path(official_lead.__file__).is_relative_to(OFFICIAL):
    raise RuntimeError("A different Medley or LEAD is imported. Restart the kernel.")
ENC = yaml.safe_load((MEDLEY_PY / "configs/encoders/lead.yaml").read_text(encoding="utf-8"))
CHANNELS = "Fp1 Fp2 F7 F3 Fz F4 F8 T7 C3 Cz C4 T8 P7 P3 Pz P4 P8 O1 O2".split()
assert ENC["channels"] == CHANNELS and ENC["pool"] == "channel"
assert (ENC["fs"], ENC["seg_sec"], ENC["amp_div"], ENC["montage"]) == (200, 2, 1.0, "standard_1005")
assert ENC["preprocess"] == {"l_freq": 0.5, "h_freq": 45.0, "rereference": "average"}
ENC.update(repo=str(OFFICIAL), weights=str(WEIGHTS))
PP = dict(flag_uv=100.0, min_good_channels=20, notch=None, **ENC["preprocess"])
print("Medley", MEDLEY_COMMIT, "LEAD", LEAD_COMMIT, "P-Base SHA256", WEIGHT_SHA)
preproc = bundle["preprocessing"]
if (preproc["channels"], preproc["fs"], preproc["seg_sec"], preproc["montage"],
        preproc["amp_div"], preproc["preprocessing"], preproc["lines"]) != (
        CHANNELS, 200, 2, "standard_1005", 1.0, PP, {"turku": 50, "unm": 60, "ucsd": 60}):
    raise ValueError("Transferred preprocessing differs from the audited baseline.")


## 3. EEG-only loaders

Memory-map EEG and apply one Medley per-window/channel Z-score. Balance subjects in training and use all evaluation windows.

In [ ]:
class Windows(Dataset):
    def __init__(self, recordings):
        self.recordings = list(recordings.itertuples(index=False))
        self.ends = list(itertools.accumulate(int(r.n_windows) for r in self.recordings))
        self.mm = {}
    def __len__(self):
        return self.ends[-1] if self.ends else 0
    def locate(self, index):
        if not 0 <= index < len(self):
            raise IndexError(index)
        i = bisect.bisect_right(self.ends, index)
        return self.recordings[i], index - (self.ends[i-1] if i else 0)
    def raw_window(self, index):
        record, offset = self.locate(index)
        if record.cache_path not in self.mm:
            self.mm[record.cache_path] = np.load(record.cache_path, mmap_mode="r", allow_pickle=False)
        return np.ascontiguousarray(self.mm[record.cache_path][offset].T)
    def __getitem__(self, index):
        record, _ = self.locate(index)
        raw = self.raw_window(index)
        normalized = np.ascontiguousarray(_zscore(raw[None])[0].T)
        return torch.from_numpy(normalized), record.label_pd, record.subject_uid
    def training_weights(self):
        person_windows = Counter()
        for record in self.recordings:
            person_windows[record.person_uid] += record.n_windows
        return torch.tensor(np.concatenate([
            np.full(record.n_windows, 1 / person_windows[record.person_uid]) for record in self.recordings]),
            dtype=torch.double)

def loader(dataset, train_seed=None):
    sampler = None
    generator = torch.Generator().manual_seed(CFG["seed"] if train_seed is None else train_seed)
    if train_seed is not None:
        sampler = WeightedRandomSampler(dataset.training_weights(), len(dataset), replacement=True,
                                        generator=generator)
    return DataLoader(dataset, batch_size=CFG["batch_size"], sampler=sampler,
                      pin_memory=CUDA, num_workers=CFG["num_workers"], generator=generator)


datasets = {name: Windows(cache_rows[cache_rows.split == name]) for name in ("train", "val", "test")}
print({name: len(dataset) for name, dataset in datasets.items()})


## 4. Differentiable model and CUDA smoke test

Check finite backward gradients, encoder updates, adapter feature parity and GPU memory. The training run reloads the initial weights.

In [ ]:
HEAD_PREFIX = ("projection_head.", "classifier.")

def checkpoint_audit(model, path):
    payload = torch.load(path, map_location="cpu", weights_only=True)
    state = payload.get("state_dict", payload)
    if not isinstance(state, dict) or not state:
        raise ValueError("Expected a nonempty model state dict.")
    cleaned = {key.removeprefix("module."): value for key, value in state.items()}
    own = model.state_dict()
    report = dict(missing=sorted(set(own) - set(cleaned)), unexpected=sorted(set(cleaned) - set(own)),
                  incompatible=[k for k in sorted(set(own) & set(cleaned))
                                if not torch.is_tensor(cleaned[k]) or cleaned[k].shape != own[k].shape])
    if len(cleaned) != len(state) or report["unexpected"] or report["incompatible"] or any(
            not k.startswith(HEAD_PREFIX) for k in report["missing"]):
        raise RuntimeError(f"Incompatible checkpoint {path}: {report}")
    return report

class PDClassifier(nn.Module):
    def __init__(self, backbone, strategy):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(2432, 2)
        self.strategy = strategy
        backbone.requires_grad_(False)
        if strategy == "full":
            for name, parameter in backbone.named_parameters():
                parameter.requires_grad_(not name.startswith(HEAD_PREFIX))
        elif strategy == "partial":
            n = CFG["last_layers"]
            layers = backbone.encoder.attn_layers
            if not 1 <= n <= len(layers):
                raise ValueError(f"last_layers must be in [1, {len(layers)}].")
            for layer in layers[-n:]:
                layer.requires_grad_(True)
        elif strategy != "head":
            raise ValueError("strategy must be full, partial, or head.")
        self.train()
    def train(self, mode=True):
        super().train(mode)
        if mode and self.strategy != "full":
            self.backbone.eval()
            if self.strategy == "partial":
                for layer in self.backbone.encoder.attn_layers[-CFG["last_layers"]:]:
                    layer.train()
        return self
    def channel_tokens(self, x):
        if x.ndim != 3 or x.shape[1:] != (400, 19) or not torch.isfinite(x).all():
            raise ValueError(f"Expected finite [B, 400, 19] EEG, got {tuple(x.shape)}.")
        route = x.new_zeros((len(x), 4))
        route[:, 2] = ENC["fs"]
        route[:, 3] = FrozenLEAD.DATASET_ID
        encoded, channels, patches, _, _ = self.backbone._encode_full(x, route, apply_augmentation=False)
        tokens = self.backbone.act(self.backbone._last_token_per_channel(encoded, channels, patches))
        if tokens.shape != (len(x), 19, 128) or patches != 8:
            raise RuntimeError(f"Unexpected token grid {tuple(tokens.shape)} and {patches} patches.")
        return tokens
    def forward(self, x):
        return self.head(self.channel_tokens(x).flatten(1))

def pretrained(strategy):
    adapter = FrozenLEAD(ENC)
    report = checkpoint_audit(adapter.model, WEIGHTS)
    return PDClassifier(adapter.model, strategy).to(DEVICE), adapter, report

def optimizer_for(model):
    return torch.optim.AdamW([
        dict(params=[p for p in model.backbone.parameters() if p.requires_grad], lr=CFG["encoder_lr"]),
        dict(params=model.head.parameters(), lr=CFG["head_lr"])],
        weight_decay=CFG["weight_decay"])

LOSS = nn.CrossEntropyLoss()
AMP = CUDA and CFG["amp"]

def smoke_test(windows):
    if CUDA:
        torch.cuda.reset_peak_memory_stats()
    seed_all(CFG["seed"])
    model, adapter, keys = pretrained("full")
    indices_by_cohort = {}
    for start, record in zip([0, *windows.ends[:-1]], windows.recordings):
        indices_by_cohort.setdefault(record.cohort, start)
    indices = list(indices_by_cohort.values())
    count = len(indices)
    x = torch.stack([windows[i][0] for i in indices]).to(DEVICE)
    labels = torch.tensor([windows[i][1] for i in indices], device=DEVICE)
    raw = np.stack([windows.raw_window(i) for i in indices])
    model.eval()
    expected = adapter.embed(raw)
    with torch.no_grad():
        observed = model.channel_tokens(x).flatten(1).cpu().numpy()
    np.testing.assert_allclose(observed, expected, rtol=2e-5, atol=2e-5)
    model.train()
    optimizer = optimizer_for(model)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    with torch.autocast(device_type=DEVICE.type, enabled=AMP):
        logits = model(x)
        loss = LOSS(logits, labels)
    if logits.shape != (count, 2) or not torch.isfinite(logits).all() or not torch.isfinite(loss):
        raise ValueError("Unexpected logits shape or non-finite smoke loss.")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    grads = [(name, p) for name, p in model.backbone.named_parameters() if p.grad is not None]
    updated = next(((name, p) for name, p in grads if torch.count_nonzero(p.grad)), None)
    if updated is None or not all(p.grad is not None and torch.count_nonzero(p.grad)
                                  for p in model.head.parameters()):
        raise RuntimeError("Encoder or head gradient is missing or zero.")
    name, parameter = updated
    before = parameter.detach().clone()
    nn.utils.clip_grad_norm_(model.parameters(), CFG["clip"], error_if_nonfinite=True)
    scaler.step(optimizer)
    scaler.update()
    if torch.equal(parameter, before) or not torch.isfinite(parameter).all():
        raise RuntimeError(f"Optimizer failed to update {name} with finite values.")
    return dict(device=str(DEVICE), cohorts=list(indices_by_cohort), logits=list(logits.shape), tokens=[count, 19, 128],
                loss=float(loss.detach()), updated_parameter=name, pretrained_keys=keys,
                parameters=sum(p.numel() for p in model.parameters()),
                trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
                peak_gpu_bytes=torch.cuda.max_memory_allocated() if CUDA else None)


smoke_report = None
smoke_report = smoke_test(datasets["train"])
write_json(PROJECT / "logs/model_smoke.json", smoke_report)
print(smoke_report)


## 5. Full fine-tuning

Default batch 32. Use batch 8 and accumulation 4 if memory is limited, before starting an experiment.

In [ ]:
CKPT = PROJECT / "checkpoints" / CFG["run"]
RESULT = PROJECT / "results" / CFG["run"]

def predict_windows(model, split):
    model.eval()
    dataset = datasets[split]
    predictions, labels, subjects = [], [], []
    total_loss = 0.0
    with torch.inference_mode():
        for x, y, ids in loader(dataset):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                logits = model(x)
            logits = logits.float()
            loss = LOSS(logits, y)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite {split} loss.")
            total_loss += loss.item() * len(y)
            predictions.extend(logits.softmax(1)[:, 1].cpu().tolist())
            labels.extend(y.cpu().tolist())
            subjects.extend(ids)
    result = pd.DataFrame(dict(subject_uid=subjects, label_pd=labels, probability=predictions))
    key = finalized[["subject_uid", "person_uid", "cohort"]]
    return total_loss / len(dataset), result.merge(key, on="subject_uid", validate="many_to_one")

def subject_predictions(windows):
    keys = ["subject_uid", "person_uid", "cohort", "label_pd"]
    return windows.groupby(keys, as_index=False).agg(probability=("probability", "mean"),
                                                     n_windows=("probability", "size"))

def save_torch(path, payload):
    atomic(path, lambda p: torch.save(payload, p))

def run_identity():
    return dict(config={k: v for k, v in CFG.items() if k != "resume"}, dependencies=versions,
                python=sys.version, torch_cuda=torch.version.cuda, medley=MEDLEY_COMMIT,
                lead=LEAD_COMMIT, pretrained_sha256=WEIGHT_SHA, transfer_manifest_sha256=TRANSFER_SHA,
                cache_keys=sorted(cache_rows.cache_key), smoke=smoke_report)

def model_snapshot(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def train_experiment():
    if not CUDA or not smoke_report or smoke_report["device"] != "cuda" or set(datasets) != {"train", "val", "test"}:
        raise RuntimeError("Full training requires CUDA, a passed CUDA smoke test, and the full cache/split.")
    if not CFG["overlap_acknowledgment"].strip():
        raise RuntimeError("Review logs/overlap_review.md and enter a written overlap acknowledgment.")
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", CFG["run"]):
        raise ValueError("Use a simple directory name for the run.")
    for key in ("batch_size", "epochs", "patience", "accumulation"):
        if not isinstance(CFG[key], int) or CFG[key] < 1:
            raise ValueError(f"{key} must be a positive integer.")
    if CFG["num_workers"] != 0:
        raise ValueError("Keep num_workers=0 for this memory-mapped notebook loader.")
    if CFG["resume"]:
        if not (CKPT / "last_model.pth").is_file() or not RESULT.is_dir():
            raise FileNotFoundError("Resume requires the existing last checkpoint and results directory.")
    elif CKPT.exists() or RESULT.exists():
        raise FileExistsError("Run directory exists. Use explicit resume or choose a new run name.")

    identity = run_identity()
    signature = digest({k: v for k, v in identity.items() if k != "smoke"})
    seed_all(CFG["seed"])
    model, _, _ = pretrained(CFG["strategy"])
    optimizer = optimizer_for(model)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    history, best, stale, start = [], None, 0, 0
    if CFG["resume"]:
        saved = torch.load(CKPT / "last_model.pth", map_location="cpu", weights_only=True)
        if saved["signature"] != signature:
            raise ValueError("Run identity changed. Restore the original inputs/configuration or use a new name.")
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        scaler.load_state_dict(saved["scaler"])
        history, best, stale, start = saved["history"], saved["best"], saved["stale"], saved["epoch"] + 1
        torch.set_rng_state(saved["cpu_rng"])
        torch.cuda.set_rng_state_all(saved["cuda_rng"])
        if best is not None:
            save_torch(CKPT / "best_model.pth", best)
            write_table(RESULT / "training_history.csv", pd.DataFrame(history), ",")
        del saved
    else:
        CKPT.mkdir()
        RESULT.mkdir()
        write_json(CKPT / "config.json", identity)
        save_torch(CKPT / "last_model.pth", dict(model=model_snapshot(model),
                   optimizer=optimizer.state_dict(), scaler=scaler.state_dict(), epoch=-1,
                   best=None, history=[], stale=0, signature=signature,
                   cpu_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all()))

    for epoch in range(start, CFG["epochs"]):
        if stale >= CFG["patience"]:
            break
        model.train()
        torch.cuda.reset_peak_memory_stats()
        train_loader = loader(datasets["train"], CFG["seed"] + epoch)
        stream = iter(train_loader)
        loss_sum = 0.0
        while batches := list(itertools.islice(stream, CFG["accumulation"])):
            count = sum(len(labels) for _, labels, _ in batches)
            optimizer.zero_grad(set_to_none=True)
            for x, y, _ in batches:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                    loss = LOSS(model(x), y)
                if not torch.isfinite(loss):
                    raise RuntimeError("Non-finite training loss.")
                scaler.scale(loss * (len(y) / count)).backward()
                loss_sum += loss.item() * len(y)
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["clip"], error_if_nonfinite=not AMP)
            scaler.step(optimizer)
            scaler.update()
        val_loss, windows = predict_windows(model, "val")
        val = subject_predictions(windows)
        auc = float(metrics.roc_auc_score(val.label_pd, val.probability))
        row = dict(epoch=epoch + 1, train_loss=loss_sum / len(datasets["train"]), val_loss=val_loss,
                   val_subject_auroc=auc, peak_gpu_bytes=torch.cuda.max_memory_allocated())
        history.append(row)
        state = model_snapshot(model)
        improved = best is None or auc > best["val_subject_auroc"]
        if improved:
            best = dict(model=state, epoch=epoch, val_subject_auroc=auc, signature=signature)
            stale = 0
        else:
            stale += 1
        save_torch(CKPT / "last_model.pth", dict(model=state, optimizer=optimizer.state_dict(),
                    scaler=scaler.state_dict(), epoch=epoch, best=best, history=history, stale=stale,
                    signature=signature, cpu_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all()))
        if improved:
            save_torch(CKPT / "best_model.pth", best)
        write_table(RESULT / "training_history.csv", pd.DataFrame(history), ",")
        print(row, flush=True)
    print(f"Best epoch {best['epoch'] + 1} with validation subject AUROC {best['val_subject_auroc']:.4f}")

def recovered_history():
    path = RESULT / "training_history.csv"
    if path.is_file():
        return pd.read_csv(path)
    saved = torch.load(CKPT / "last_model.pth", map_location="cpu", weights_only=True)
    rows = saved.get("history")
    if not isinstance(rows, list) or not rows or len(rows) != saved["epoch"] + 1:
        raise ValueError("No complete epoch history in the last checkpoint.")
    frame = pd.DataFrame(rows)
    write_table(path, frame, ",")
    return frame

def load_best():
    if not CUDA or "test" not in datasets:
        raise RuntimeError("Load the same CUDA run configuration and full data before evaluation/export.")
    best = torch.load(CKPT / "best_model.pth", map_location="cpu", weights_only=True)
    identity = run_identity()
    if best["signature"] != digest({k: v for k, v in identity.items() if k != "smoke"}):
        raise ValueError("The best checkpoint belongs to different inputs or settings.")
    last = torch.load(CKPT / "last_model.pth", map_location="cpu", weights_only=True)
    if last["stale"] < CFG["patience"] and last["epoch"] + 1 < CFG["epochs"]:
        raise RuntimeError("Finish or resume training before opening the held-out test set.")
    if best["epoch"] != last["best"]["epoch"]:
        raise RuntimeError("Resume to reconcile best and last checkpoints.")
    model, _, _ = pretrained(CFG["strategy"])
    model.load_state_dict(best["model"], strict=True)
    return model.eval(), best

## Run training explicitly

Set RESUME=True with the same ZIP, parameters and run directory on restart. A Drive-backed OUTPUT_ROOT saves each completed epoch to persistent storage.

In [ ]:
# train_experiment()


## 6. Evaluate the held-out split

Run this section after training completes. It reports held-out test metrics at both levels, with subject metrics primary. Cohort AUROC is undefined when only one class is present. Small test cohorts, acquisition hardware, EO/EC composition, medication protocols, and shared cohort shortcuts limit interpretation. Individual windows inherit the participant label even when the short segment has little disease-related signal. PD/HC discrimination alone cannot establish an effect on MoCA prediction.

In [ ]:
def classification_metrics(frame):
    y, p = frame.label_pd.to_numpy(), frame.probability.to_numpy()
    prediction = (p >= CFG["threshold"]).astype(int)
    cm = metrics.confusion_matrix(y, prediction, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel().tolist()
    both = len(np.unique(y)) == 2
    sensitivity = tp / (tp + fn) if tp + fn else None
    specificity = tn / (tn + fp) if tn + fp else None
    return dict(n=len(y), hc=tn + fp, pd=tp + fn, accuracy=float(metrics.accuracy_score(y, prediction)),
                balanced_accuracy=(sensitivity + specificity) / 2 if both else None,
                precision=tp / (tp + fp) if tp + fp else 0.0, sensitivity=sensitivity,
                specificity=specificity, f1=float(metrics.f1_score(y, prediction, zero_division=0)),
                auroc=float(metrics.roc_auc_score(y, p)) if both else None, confusion_matrix=cm.tolist())

def evaluate_experiment():
    model, best = load_best()
    loss, windows = predict_windows(model, "test")
    subjects = subject_predictions(windows)
    subject_scores = classification_metrics(subjects)
    report = dict(test_loss=loss, best_epoch=best["epoch"] + 1, threshold=CFG["threshold"],
                  window=classification_metrics(windows), subject=subject_scores)
    cohort_scores = [dict(cohort=cohort, level=level, **classification_metrics(group))
                     for level, table in (("subject", subjects), ("window", windows))
                     for cohort, group in table.groupby("cohort")]
    write_json(RESULT / "test_metrics.json", report)
    write_table(RESULT / "cohort_metrics.csv", pd.DataFrame(cohort_scores), ",")
    write_table(RESULT / "test_subject_predictions_private.tsv",
                subjects.assign(predicted_label=(subjects.probability >= CFG["threshold"]).astype(int)))
    history = recovered_history()

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
    for field, label in (("train_loss", "Train"), ("val_loss", "Validation")):
        axes[0].plot(history.epoch, history[field], label=label)
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy")
    axes[0].legend()
    axes[1].plot(history.epoch, history.val_subject_auroc)
    axes[1].axvline(best["epoch"] + 1, linestyle="--", color="gray")
    axes[1].set(xlabel="Epoch", ylabel="Validation subject AUROC", ylim=(0, 1))
    fig.savefig(RESULT / "learning_curves.png", dpi=160)
    plt.close(fig)

    display = metrics.ConfusionMatrixDisplay(
        confusion_matrix=np.array(subject_scores["confusion_matrix"]), display_labels=["HC", "PD"])
    display.plot()
    display.ax_.set_title("Held-out subjects")
    display.figure_.savefig(RESULT / "confusion_matrix.png", dpi=160, bbox_inches="tight")
    plt.close(display.figure_)
    fig, ax = plt.subplots()
    for label, table in (("Subject", subjects), ("Window", windows)):
        metrics.RocCurveDisplay.from_predictions(table.label_pd, table.probability, ax=ax, name=label)
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set_title("Held-out ROC")
    fig.savefig(RESULT / "roc_curve.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(json.dumps(report, indent=2))

# evaluate_experiment()

## 7. Export and verify with the frozen adapter

Export the best encoder and verify [19, 128] channel features and 2432-dimensional Medley features. For later use, set repo and weights in the YAML to the destination machine's paths.

In [ ]:
def export_encoder():
    model, best = load_best()
    target = CKPT / "lead_pd_finetuned.pth"
    target_yaml = CKPT / "lead_pd_finetuned.yaml"
    verification = CKPT / "verification.json"
    if any(p.exists() for p in (target, target_yaml, verification)):
        raise FileExistsError("An export exists in this run directory.")
    staging = Path("/content") / f".lead-export-{uuid.uuid4().hex}.tmp"
    export_config = dict(ENC, name="lead_pd_finetuned", weights=str(target))
    try:
        # This is the pretrain-form official module used by Medley, including
        # its unused projection head. The separate PDClassifier.head is absent.
        save_torch(staging, model_snapshot(model.backbone))
        keys = checkpoint_audit(model.backbone, staging)
        if keys["missing"]:
            raise RuntimeError(f"Export lost parameters: {keys['missing']}")
        adapter = FrozenLEAD(dict(export_config, weights=str(staging)))
        if any(p.requires_grad for p in adapter.model.parameters()) or adapter.model.training:
            raise RuntimeError("Export adapter failed to freeze the encoder.")
        dataset = datasets["train"]
        sample_indices = [i for i in range(min(4, len(dataset)))
                          if dataset.locate(i)[0].subject_uid == dataset.locate(0)[0].subject_uid]
        eeg = np.stack([dataset.raw_window(i) for i in sample_indices])
        x = torch.stack([dataset[i][0] for i in sample_indices]).to(DEVICE)
        with torch.inference_mode():
            channels = model.channel_tokens(x).cpu().numpy()
        features = adapter.embed(eeg)
        np.testing.assert_allclose(features, channels.reshape(len(x), 2432), rtol=2e-5, atol=2e-5)
        baseline = FrozenLEAD(ENC).embed(eeg)
        subject_vector = features.mean(axis=0)
        if features.shape != baseline.shape or channels.shape[1:] != (19, 128) or subject_vector.shape != (2432,):
            raise RuntimeError("Feature dimensions differ from the baseline.")
        report = dict(best_epoch=best["epoch"] + 1, best_validation_auroc=best["val_subject_auroc"],
                      window_channels=list(channels.shape[1:]), window_features=features.shape[1],
                      subject_features=len(subject_vector), baseline_window_features=baseline.shape[1],
                      state_dict_keys=keys, checkpoint_sha256=sha_file(staging), frozen=True,
                      medley=MEDLEY_COMMIT, lead=LEAD_COMMIT, smoke=smoke_report)
        atomic(target, lambda temporary: shutil.copyfile(staging, temporary))
        atomic(target_yaml, lambda p: p.write_text(yaml.safe_dump(export_config, sort_keys=False), encoding="utf-8"))
        write_json(verification, report)
        print(report)
    finally:
        staging.unlink(missing_ok=True)

# export_encoder()

## 8. Save before the runtime terminates

To persist every epoch, mount Drive and choose a private OUTPUT_ROOT under /content/drive/MyDrive before training. Alternatively, enable the download cell. To resume from a downloaded run, restore the checkpoints and results directory structure under OUTPUT_ROOT and set RESUME=True. Keep individual predictions private.

In [ ]:

def download_run():
    archive = Path("/content") / f"{CFG['run']}_outputs.zip"
    def write_run(temporary):
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as z:
            for directory in (CKPT, RESULT):
                for path in sorted(directory.iterdir()):
                    if path.is_file():
                        z.write(path, f"{directory.parent.name}/{directory.name}/{path.name}")
    if archive.exists():
        raise FileExistsError(archive)
    atomic(archive, write_run)
    files.download(str(archive))

# download_run()
